<h2 style="color:#4A148C; margin-bottom:6px;">
  06 – RoBERTa Base for Multi-Label Emotion Classification
</h2>

<p style="font-size:16px; margin-top:4px;">
  <strong>Overview:</strong>
  This notebook fine-tunes <strong>RoBERTa Base</strong> for multi-label
  emotion classification on short English text. RoBERTa is an improved
  variant of BERT, trained on more data with optimized pretraining
  strategies.
</p>

<p style="font-size:16px;">
  <strong>Objective:</strong>
  To predict the presence of five emotions —
  <em>anger, fear, joy, sadness,</em> and <em>surprise</em> — for each
  text sample, and to compare RoBERTa's performance against other models
  such as BERT, BiLSTM, and MLP baselines using the Macro F1-score.
</p>

<p style="font-size:16px;">
  The model uses the <code>roberta-base</code> encoder followed by a
  dropout layer and a linear classification head, with a sigmoid
  activation applied during evaluation for multi-label prediction.
  Validation performance is monitored to perform early stopping and
  select the best checkpoint, which is later used in a separate
  inference-only notebook.
</p>


In [2]:
#wandb login check to make sure everything goes smooth

from kaggle_secrets import UserSecretsClient
import wandb

# Load API key from Kaggle Secrets
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

# Login to wandb
wandb.login(key=api_key)

print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sharmilam-official (sharmila-m) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful!


In [3]:
# 06a – RoBERTa Base Training for Multi-Label Emotion Classification

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm
import wandb

# -------------------------
# 0. Config & WandB Init
# -------------------------
LABELS = ["anger", "fear", "joy", "sadness", "surprise"]

wandb.init(
    project="24ds2000048-t32025",
    entity="sharmila-m",
    name="06a_roberta_base_training",
    config={
        "model_name": "roberta-base",
        "max_len": 80,
        "batch_size": 16,
        "lr": 2e-5,
        "epochs": 5,
        "val_split": 0.1,
        "threshold": 0.5,
        "patience": 3,
        "random_state": 42
    }
)

config = wandb.config
MAX_LEN = config.max_len

torch.manual_seed(config.random_state)
np.random.seed(config.random_state)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 1. Data Loading
# -------------------------
train_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/train.csv")
test_df = pd.read_csv("/kaggle/input/2025-sep-dl-gen-ai-project/test.csv")

tokenizer = RobertaTokenizer.from_pretrained(config.model_name)

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, mode="train"):
        self.df = df
        self.tokenizer = tokenizer
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df.iloc[idx]["text"])
        encoding = self.tokenizer(
            text,
            max_length=MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        if self.mode == "train":
            labels = torch.tensor(
                self.df.iloc[idx][LABELS].values.astype("float32")
            )
            item["labels"] = labels
        return item

train_df_split, val_df_split = train_test_split(
    train_df,
    test_size=config.val_split,
    random_state=config.random_state
)

train_dataset = EmotionDataset(train_df_split, tokenizer, mode="train")
val_dataset   = EmotionDataset(val_df_split, tokenizer, mode="train")

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=config.batch_size)

# -------------------------
# 2. RoBERTa Model
# -------------------------
class RobertaForMultiLabel(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.pooler_output  # [CLS]-like pooled representation
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return logits  # raw logits

model = RobertaForMultiLabel(config.model_name, len(LABELS)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.lr)

# -------------------------
# 3. Training Loop with Early Stopping
# -------------------------
best_val_f1 = -1.0
patience = config.patience
wait = 0
best_model_path = "best_roberta_model.pt"

for epoch in range(config.epochs):
    model.train()
    running_loss = 0.0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.epochs} - Training"):
        input_ids = batch["input_ids"].to(device)
        attn = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attn)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    val_labels, val_preds = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{config.epochs} - Validation"):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attn)
            probs = torch.sigmoid(logits).cpu().numpy()

            val_preds.append(probs)
            val_labels.append(labels.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_labels = np.vstack(val_labels)
    pred_bin = (val_preds > config.threshold).astype(int)

    val_f1_macro = f1_score(val_labels, pred_bin, average="macro")
    val_accuracy = accuracy_score(val_labels, pred_bin)

    print(
        f"Epoch {epoch+1}/{config.epochs} "
        f"- Train Loss: {avg_train_loss:.4f} "
        f"- Val Macro F1: {val_f1_macro:.4f} "
        f"- Val Accuracy: {val_accuracy:.4f}"
    )

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_f1_macro": val_f1_macro,
        "val_accuracy": val_accuracy
    })

    # Early stopping logic
    if val_f1_macro > best_val_f1:
        best_val_f1 = val_f1_macro
        wait = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping.")
            break

print("Best validation Macro F1:", best_val_f1)

# -------------------------
# 4. Upload Best Model to KaggleHub
# -------------------------
import kagglehub

KAGGLE_USERNAME = "sharmilamamani"  # your Kaggle username
MODEL = "emotion-roberta"
FRAMEWORK = "pytorch"
VARIATION = "roberta-base-v1"

handle = f"{KAGGLE_USERNAME}/{MODEL}/{FRAMEWORK}/{VARIATION}"

try:
    print("Uploading RoBERTa model to KaggleHub...")
    repo = kagglehub.model_upload(
        handle,
        best_model_path,
        version_notes="RoBERTa base fine-tuned for multi-label emotion classification"
    )
    print("KaggleHub upload successful.")
    print("Handle:", handle)
    print("Repo info:", repo)
except Exception as e:
    print("KaggleHub upload failed.")
    print(type(e).__name__, ":", e)

wandb.finish()


2025-11-29 07:40:45.045960: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1764402045.192402      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1764402045.237982      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Epoch 5/5 - Train Loss: 0.0931 - Val Macro F1: 0.8543 - Val Accuracy: 0.6911
Best validation Macro F1: 0.8558317650150633
Uploading RoBERTa model to KaggleHub...
Uploading Model https://www.kaggle.com/models/sharmilamamani/emotion-roberta/pytorch/roberta-base-v1 ...
Model 'emotion-roberta' does not exist or access is forbidden for user 'sharmilamamani'. Creating or handling Model...
Model 'emotion-roberta' Created.
Starting upload for file best_roberta_model.pt


Uploading: 100%|██████████| 499M/499M [00:03<00:00, 130MB/s]  

Upload successful: best_roberta_model.pt (476MB)


Your model instance has been created.
Files are being processed...
See at: https://www.kaggle.com/models/sharmilamamani/emotion-roberta/pytorch/roberta-base-v1
KaggleHub upload successful.
Handle: sharmilamamani/emotion-roberta/pytorch/roberta-base-v1
Repo info: None


epoch,▁▃▅▆█
train_loss,█▅▃▂▁
val_accuracy,▁▅▆██
val_f1_macro,▁▅▇██
epoch,5
train_loss,0.09313
val_accuracy,0.69107
val_f1_macro,0.85428


<hr/>

<h3 style="color:#D35400;">
  Results & Runtime Summary
</h3>

<p style="font-size:16px;">
  <strong>Model:</strong> RoBERTa Base fine-tuned for multi-label emotion classification
</p>

<p style="font-size:16px;">
  <strong>Final Training Epoch:</strong> 5<br/>
  Train Loss: <strong>0.09313</strong>
</p>

<p style="font-size:16px;">
  <strong>Validation Metrics (Internal):</strong><br/>
  Macro F1 Score: <strong>0.85428</strong><br/>
  Accuracy: <strong>0.69107</strong>
</p>

<p style="font-size:16px;">
  <strong>Kaggle Public Leaderboard Score:</strong><br/>
  Macro F1 Score: <strong>0.839</strong>
</p>

<p style="font-size:16px;">
  <strong>Runtime Environment:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li><strong>Platform:</strong> Kaggle Notebook</li>
  <li><strong>Hardware:</strong> GPU (NVIDIA T4 &times; 2)</li>
  <li><strong>Estimated Runtime:</strong> ~8–10 minutes (including training and validation)</li>
  <li><strong>Frameworks:</strong> PyTorch, HuggingFace Transformers</li>
</ul>

<p style="font-size:16px;">
  <strong>Observations:</strong>
</p>

<ul style="font-size:16px; line-height:1.6;">
  <li>
    RoBERTa achieves strong validation performance with a Macro F1 score
    slightly above 0.85, confirming that pretrained transformer models
    are well-suited for emotion classification.
  </li>
  <li>
    The Kaggle leaderboard score (0.839) is closely aligned with the
    internal validation result, indicating good generalization to unseen
    test data.
  </li>
  <li>
    Compared to traditional baselines and from-scratch neural networks,
    RoBERTa provides a significant performance boost while sharing a
    similar fine-tuning pipeline with BERT, enabling a fair comparison
    between the two pretrained backbones.
  </li>
</ul>
